
# Задание Ultra Pro — PyTorch

Используется база изображений автомобилей. В файле есть два варианта модели:

1. обычная сверточная сеть на `torch.nn`;
2. модель с переносом обучения на базе `MobileNetV2` из `torchvision`.

Для достижения точности выше 93% обычно нужен второй вариант: сначала обучение только классификатора, затем дообучение последних блоков `MobileNetV2`.


In [ ]:

# Загрузка и распаковка базы
import os
import shutil
import zipfile

try:
    import gdown
except ImportError:
    !pip -q install gdown
    import gdown

ZIP_PATH = '/content/middle_fmr.zip'
IMAGE_PATH = '/content/cars'

if not os.path.exists(ZIP_PATH):
    gdown.download(
        'https://storage.yandexcloud.net/aiueducation/Content/base/l5/middle_fmr.zip',
        ZIP_PATH,
        quiet=False
    )

if os.path.exists(IMAGE_PATH):
    shutil.rmtree(IMAGE_PATH)

os.makedirs(IMAGE_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(IMAGE_PATH)

print('База распакована в:', IMAGE_PATH)
print('Содержимое папки:', os.listdir(IMAGE_PATH)[:10])


In [ ]:

# Импорты
import random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# Фиксация случайности для повторяемости результата
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', DEVICE)

if DEVICE.type == 'cpu':
    print('Предупреждение: обучение на CPU будет заметно дольше. В Colab лучше включить GPU.')


In [ ]:

# Проверка классов
CLASS_LIST = sorted([
    name for name in os.listdir(IMAGE_PATH)
    if os.path.isdir(os.path.join(IMAGE_PATH, name))
])
CLASS_COUNT = len(CLASS_LIST)

print('Количество классов:', CLASS_COUNT)
print('Классы:', CLASS_LIST)

if CLASS_COUNT == 0:
    raise ValueError('Классы не найдены. Проверьте путь IMAGE_PATH и структуру архива.')


In [ ]:

# Создание загрузчиков данных

def make_loaders(image_size=224, batch_size=32, imagenet_norm=True):
    """Создает обучающий и проверочный DataLoader с одинаковым разбиением по классам."""

    if imagenet_norm:
        normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    else:
        normalize = transforms.Normalize(
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5]
        )

    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        normalize,
    ])

    val_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        normalize,
    ])

    train_full = datasets.ImageFolder(IMAGE_PATH, transform=train_transform)
    val_full = datasets.ImageFolder(IMAGE_PATH, transform=val_transform)

    labels = [label for _, label in train_full.samples]
    indices = np.arange(len(labels))

    train_idx, val_idx = train_test_split(
        indices,
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )

    train_dataset = Subset(train_full, train_idx)
    val_dataset = Subset(val_full, val_idx)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=(DEVICE.type == 'cuda')
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=(DEVICE.type == 'cuda')
    )

    print('Обучающая выборка:', len(train_dataset))
    print('Проверочная выборка:', len(val_dataset))
    print('Классы:', train_full.classes)

    return train_loader, val_loader, train_full.classes


In [ ]:

# Функции обучения, проверки и построения графиков

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            predicted = outputs.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, patience=5, save_path=None):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    best_val_acc = 0.0
    best_state = None
    wait = 0

    history = {
        'loss': [],
        'accuracy': [],
        'val_loss': [],
        'val_accuracy': []
    }

    model.to(DEVICE)

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate_model(model, val_loader, criterion)

        history['loss'].append(train_loss)
        history['accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        print(
            f'Эпоха {epoch:02d}/{epochs} | '
            f'loss: {train_loss:.4f} | acc: {train_acc:.4f} | '
            f'val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}'
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print('Ранняя остановка: качество на проверке перестало расти.')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    if save_path is not None:
        torch.save(model.state_dict(), save_path)
        print('Модель сохранена:', save_path)

    print('Лучшая точность на проверке:', round(best_val_acc, 4))
    return history


def show_graphs(history, title=''):
    epochs = range(1, len(history['accuracy']) + 1)

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['accuracy'], label='Обучение')
    plt.plot(epochs, history['val_accuracy'], label='Проверка')
    plt.xlabel('Эпоха')
    plt.ylabel('Точность')
    plt.title('Точность ' + title)
    plt.legend()
    plt.grid()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['loss'], label='Ошибка на обучении')
    plt.plot(epochs, history['val_loss'], label='Ошибка на проверке')
    plt.xlabel('Эпоха')
    plt.ylabel('Ошибка')
    plt.title('Ошибка ' + title)
    plt.legend()
    plt.grid()

    plt.show()


In [ ]:

# Обычная сверточная сеть на PyTorch
class CarsCNN(nn.Module):
    def __init__(self, class_count):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, class_count)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:

# Обучение обычной CNN
# Этот вариант нужен для сравнения. Для 93% обычно лучше использовать MobileNetV2 ниже.

train_loader, val_loader, classes = make_loaders(
    image_size=128,
    batch_size=32,
    imagenet_norm=False
)

cnn_model = CarsCNN(class_count=len(classes))

cnn_history = train_model(
    cnn_model,
    train_loader,
    val_loader,
    epochs=15,
    lr=1e-3,
    patience=5,
    save_path='/content/cars_cnn_pytorch.pth'
)

show_graphs(cnn_history, title='CNN')


In [ ]:

# MobileNetV2 на PyTorch
class CarsMobileNet(nn.Module):
    def __init__(self, class_count):
        super().__init__()

        try:
            weights = models.MobileNet_V2_Weights.DEFAULT
            self.base_model = models.mobilenet_v2(weights=weights)
        except Exception:
            self.base_model = models.mobilenet_v2(pretrained=True)

        in_features = self.base_model.classifier[1].in_features

        self.base_model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, class_count)
        )

    def freeze_base(self):
        for param in self.base_model.features.parameters():
            param.requires_grad = False

    def unfreeze_last_blocks(self, blocks_count=4):
        for param in self.base_model.features.parameters():
            param.requires_grad = False

        for block in self.base_model.features[-blocks_count:]:
            for param in block.parameters():
                param.requires_grad = True

        for param in self.base_model.classifier.parameters():
            param.requires_grad = True

    def forward(self, x):
        return self.base_model(x)


In [ ]:

# Этап 1. Обучение только классификатора MobileNetV2

train_loader, val_loader, classes = make_loaders(
    image_size=224,
    batch_size=64,
    imagenet_norm=True
)

mobilenet_model = CarsMobileNet(class_count=len(classes))
mobilenet_model.freeze_base()

mobilenet_history_1 = train_model(
    mobilenet_model,
    train_loader,
    val_loader,
    epochs=20,
    lr=3e-4,
    patience=5,
    save_path='/content/cars_mobilenet_stage1_pytorch.pth'
)

show_graphs(mobilenet_history_1, title='MobileNetV2: этап 1')


In [ ]:

# Этап 2. Дообучение последних блоков MobileNetV2
# Запускается после этапа 1.

mobilenet_model.unfreeze_last_blocks(blocks_count=4)

mobilenet_history_2 = train_model(
    mobilenet_model,
    train_loader,
    val_loader,
    epochs=25,
    lr=5e-5,
    patience=5,
    save_path='/content/cars_mobilenet_final_pytorch.pth'
)

show_graphs(mobilenet_history_2, title='MobileNetV2: дообучение')


In [ ]:

# Сохранение финальной модели вместе со списком классов
FINAL_MODEL_PATH = '/content/cars_mobilenet_final_full.pth'

torch.save({
    'model_state_dict': mobilenet_model.state_dict(),
    'classes': classes,
    'image_size': 224
}, FINAL_MODEL_PATH)

print('Финальная модель сохранена:', FINAL_MODEL_PATH)
print('Классы:', classes)


In [ ]:

# Проверка модели на одном изображении из проверочной выборки
mobilenet_model.eval()

images, labels = next(iter(val_loader))
image = images[0].unsqueeze(0).to(DEVICE)
true_label = labels[0].item()

with torch.no_grad():
    output = mobilenet_model(image)
    predicted_label = output.argmax(dim=1).item()

print('Истинный класс:', classes[true_label])
print('Предсказанный класс:', classes[predicted_label])
